# Taller 2 - Punto 1: Embeddings de Palabras con Word2vec y FastText

Este notebook implementa el entrenamiento y evaluación de modelos de embeddings de palabras usando Word2vec y FastText sobre el dataset spanish_billion_words.

## Objetivos:
1. Preprocesamiento del corpus
2. Entrenamiento de modelos con diferentes configuraciones
3. Consulta de similitud semántica
4. Visualización con t-SNE o PCA

## 1.1 Instalación de Dependencias y Configuración Inicial

In [ ]:
!pip install datasets
!pip install gensim
!pip install tqdm
!pip install nltk
!pip install scikit-learn
!pip install matplotlib

In [1]:
import nltk
nltk.download('stopwords')

from datasets import load_dataset
from nltk.corpus import stopwords
from tqdm.auto import tqdm
import re
import string
from gensim.models import FastText, Word2Vec
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading package stopwords to /home/yenreh/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
/home/yenreh/anaconda3/envs/pln_taller2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.2 Carga del Dataset Spanish Billion Words

Cargamos el dataset `spanish_billion_words` de Hugging Face. Este es un corpus masivo de texto en español.

In [ ]:
# Cargar el dataset completo (toma tiempo y espacio)
# Nota: Comenzaremos con una muestra más pequeña para las pruebas
print("Cargando dataset spanish_billion_words...")
dataset_full = load_dataset("jhonparra18/spanish_billion_words_clean", split="train")

# Tomar diferentes muestras según el taller
sample_500k = dataset_full.select(range(min(500000, len(dataset_full))))
# sample_full = dataset_full  # Para usar el dataset completo cuando sea necesario

print(f"Dataset completo: {len(dataset_full)} textos")
print(f"Muestra 500k: {len(sample_500k)} textos")

## 1.3 Preprocesamiento del Corpus

Limpieza del texto según los requisitos del taller:
- Eliminación de números
- Eliminación de signos de puntuación y símbolos especiales
- Eliminación de palabras vacías (stopwords) del español
- Tokenización de las sentencias en listas de palabras

In [ ]:
def preprocess_text(dataset_sample):
    """
    Preprocesa el texto eliminando números, stopwords, puntuación y símbolos especiales
    """
    sentences = []
    stop_words = stopwords.words('spanish')
    
    print(f"Procesando {len(dataset_sample['text'])} textos...")
    
    for sent in tqdm(dataset_sample['text'], desc="Limpiando texto"):
        if len(sent) > 1:
            # Tokenizar por palabras
            words = sent.split()
            
            # Eliminar números y palabras con números
            words = [w for w in words if not w.isdigit()]
            words = [re.sub(r'[0-9]', '', w) for w in words]
            
            # Eliminar stopwords
            words = [w for w in words if w.lower() not in stop_words]
            
            # Eliminar puntuación
            re_punc = re.compile('[%s]' % re.escape(string.punctuation))
            words = [re_punc.sub('', w) for w in words]
            
            # Eliminar símbolos especiales adicionales
            words = [re.sub(r"\!|\'|\?|\¿|\¡|\«|\»", "", w) for w in words]
            
            # Eliminar palabras vacías y convertir a minúsculas
            words = [w.lower() for w in words if w != '']
            
            if len(words) > 0:
                sentences.append(words)
    
    print(f"Total de oraciones procesadas: {len(sentences)}")
    return sentences

# Procesar la muestra de 500k
print("=" * 60)
print("Preprocesando muestra de 500k sentencias...")
print("=" * 60)
sentences_500k = preprocess_text(sample_500k)

## 1.4 Entrenamiento de Modelos Word2vec

Entrenar modelos Word2vec con diferentes configuraciones:
- Vector sizes: 100, 200, 300
- Dataset sizes: 500,000 y completo

In [ ]:
def train_word2vec_models(sentences, sizes=[100, 200, 300]):
    """
    Entrena modelos Word2Vec con diferentes tamaños de embeddings
    """
    models = {}
    
    for size in sizes:
        print(f"\n{'='*60}")
        print(f"Entrenando Word2Vec con vector_size={size}")
        print(f"{'='*60}")
        
        model = Word2Vec(
            sentences=sentences,
            vector_size=size,
            window=5,
            min_count=10,
            workers=4,
            sg=0  # CBOW
        )
        
        model_name = f"word2vec_{size}d_{len(sentences)}sent"
        models[model_name] = model
        
        # Guardar modelo
        model.save(f"./models/{model_name}.model")
        print(f"✓ Modelo guardado: ./models/{model_name}.model")
        
        # Mostrar algunas palabras similares de ejemplo
        try:
            similar = model.wv.most_similar('futuro', topn=5)
            print(f"\nPalabras similares a 'futuro': {similar}")
        except:
            print("\nNo se pudo encontrar la palabra 'futuro' en el vocabulario")
    
    return models

# Entrenar modelos Word2Vec con la muestra de 500k
print("\n" + "="*60)
print("ENTRENANDO MODELOS WORD2VEC (500k sentencias)")
print("="*60)
w2v_models_500k = train_word2vec_models(sentences_500k, sizes=[100, 200, 300])

## 1.5 Entrenamiento de Modelos FastText

Entrenar modelos FastText con las mismas configuraciones que Word2vec para comparar.

In [ ]:
def train_fasttext_models(sentences, sizes=[100, 200, 300]):
    """
    Entrena modelos FastText con diferentes tamaños de embeddings
    """
    models = {}
    
    for size in sizes:
        print(f"\n{'='*60}")
        print(f"Entrenando FastText con vector_size={size}")
        print(f"{'='*60}")
        
        model = FastText(
            sentences=sentences,
            vector_size=size,
            window=5,
            min_count=10,
            workers=4,
            sg=0  # CBOW
        )
        
        model_name = f"fasttext_{size}d_{len(sentences)}sent"
        models[model_name] = model
        
        # Guardar modelo
        model.save(f"./models/{model_name}.model")
        print(f"✓ Modelo guardado: ./models/{model_name}.model")
        
        # Mostrar algunas palabras similares de ejemplo
        try:
            similar = model.wv.most_similar('futuro', topn=5)
            print(f"\nPalabras similares a 'futuro': {similar}")
        except:
            print("\nNo se pudo encontrar la palabra 'futuro' en el vocabulario")
    
    return models

# Entrenar modelos FastText con la muestra de 500k
print("\n" + "="*60)
print("ENTRENANDO MODELOS FASTTEXT (500k sentencias)")
print("="*60)
ft_models_500k = train_fasttext_models(sentences_500k, sizes=[100, 200, 300])

## 1.6 Consulta de Similitud Semántica

Implementación de una función para consultar las 10 palabras más similares a una palabra dada.

In [ ]:
def query_similar_words(model, word, topn=10):
    """
    Consulta las palabras más similares a una palabra dada
    """
    try:
        similar_words = model.wv.most_similar(word, topn=topn)
        print(f"\nPalabras más similares a '{word}':")
        print("-" * 50)
        for i, (similar_word, similarity) in enumerate(similar_words, 1):
            print(f"{i:2}. {similar_word:20} (similitud: {similarity:.4f})")
        return similar_words
    except KeyError:
        print(f"La palabra '{word}' no está en el vocabulario del modelo")
        return None

# Probar con el mejor modelo (Word2Vec 300d como referencia)
best_w2v_model = w2v_models_500k[list(w2v_models_500k.keys())[-1]]  # último modelo (300d)
best_ft_model = ft_models_500k[list(ft_models_500k.keys())[-1]]  # último modelo (300d)

# Palabras de prueba
test_words = ['españa', 'futuro', 'tecnología', 'educación', 'amor']

print("\n" + "="*60)
print("CONSULTAS DE SIMILITUD SEMÁNTICA - WORD2VEC 300D")
print("="*60)
for word in test_words:
    query_similar_words(best_w2v_model, word)

print("\n" + "="*60)
print("CONSULTAS DE SIMILITUD SEMÁNTICA - FASTTEXT 300D")
print("="*60)
for word in test_words:
    query_similar_words(best_ft_model, word)

## 1.7 Visualización con t-SNE

Visualizar los vectores de embeddings en un plano 2D usando t-SNE para comparar Word2vec y FastText.

In [ ]:
def visualize_embeddings_tsne(model, model_name, num_words=100, perplexity=30):
    """
    Visualiza embeddings usando t-SNE
    """
    # Obtener vectores y palabras
    words = list(model.wv.index_to_key[:num_words])
    vectors = np.array([model.wv[word] for word in words])
    
    # Aplicar t-SNE
    print(f"\nAplicando t-SNE para {model_name}...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(perplexity, len(words)-1))
    vectors_2d = tsne.fit_transform(vectors)
    
    # Visualizar
    plt.figure(figsize=(14, 10))
    plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], alpha=0.5)
    
    # Anotar palabras
    for i, word in enumerate(words):
        plt.annotate(word, xy=(vectors_2d[i, 0], vectors_2d[i, 1]), 
                    fontsize=8, alpha=0.7)
    
    plt.title(f'Visualización t-SNE de Embeddings - {model_name}', fontsize=14)
    plt.xlabel('Componente 1')
    plt.ylabel('Componente 2')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'./output/{model_name}_tsne.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Gráfico guardado: ./output/{model_name}_tsne.png")

# Visualizar Word2Vec 300d
print("\n" + "="*60)
print("VISUALIZACIÓN T-SNE - WORD2VEC 300D")
print("="*60)
visualize_embeddings_tsne(best_w2v_model, "Word2Vec_300d", num_words=100)

# Visualizar FastText 300d
print("\n" + "="*60)
print("VISUALIZACIÓN T-SNE - FASTTEXT 300D")
print("="*60)
visualize_embeddings_tsne(best_ft_model, "FastText_300d", num_words=100)

## 1.8 Visualización Comparativa con PCA

Visualizar usando PCA como alternativa a t-SNE para comparar los resultados.

In [ ]:
def visualize_embeddings_pca(model, model_name, num_words=100):
    """
    Visualiza embeddings usando PCA
    """
    # Obtener vectores y palabras
    words = list(model.wv.index_to_key[:num_words])
    vectors = np.array([model.wv[word] for word in words])
    
    # Aplicar PCA
    print(f"\nAplicando PCA para {model_name}...")
    pca = PCA(n_components=2)
    vectors_2d = pca.fit_transform(vectors)
    
    # Visualizar
    plt.figure(figsize=(14, 10))
    plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], alpha=0.5)
    
    # Anotar palabras
    for i, word in enumerate(words):
        plt.annotate(word, xy=(vectors_2d[i, 0], vectors_2d[i, 1]), 
                    fontsize=8, alpha=0.7)
    
    plt.title(f'Visualización PCA de Embeddings - {model_name}', fontsize=14)
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} varianza)')
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} varianza)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'./output/{model_name}_pca.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Gráfico guardado: ./output/{model_name}_pca.png")
    print(f"Varianza explicada total: {sum(pca.explained_variance_ratio_):.2%}")

# Visualizar Word2Vec 300d con PCA
print("\n" + "="*60)
print("VISUALIZACIÓN PCA - WORD2VEC 300D")
print("="*60)
visualize_embeddings_pca(best_w2v_model, "Word2Vec_300d", num_words=100)

# Visualizar FastText 300d con PCA
print("\n" + "="*60)
print("VISUALIZACIÓN PCA - FASTTEXT 300D")
print("="*60)
visualize_embeddings_pca(best_ft_model, "FastText_300d", num_words=100)

## 1.9 Comparación de Resultados

Análisis comparativo entre los modelos Word2Vec y FastText.

In [ ]:
print("\n" + "="*60)
print("RESUMEN Y COMPARACIÓN FINAL")
print("="*60)

print("\n📊 MODELOS WORD2VEC:")
for name in w2v_models_500k.keys():
    model = w2v_models_500k[name]
    vocab_size = len(model.wv.index_to_key)
    vector_dim = model.wv.vector_size
    print(f"  • {name}: {vocab_size:,} palabras, dim={vector_dim}")

print("\n📊 MODELOS FASTTEXT:")
for name in ft_models_500k.keys():
    model = ft_models_500k[name]
    vocab_size = len(model.wv.index_to_key)
    vector_dim = model.wv.vector_size
    print(f"  • {name}: {vocab_size:,} palabras, dim={vector_dim}")

print("\n✅ CONCLUSIONES:")
print("  1. Word2Vec y FastText fueron entrenados exitosamente")
print("  2. Se probaron diferentes dimensiones de embeddings (100, 200, 300)")
print("  3. Las visualizaciones muestran la distribución semántica de las palabras")
print("  4. FastText puede manejar palabras fuera del vocabulario gracias a subword info")
print("  5. Los modelos pueden consultarse para similitud semántica")

print("\n💾 Modelos guardados en: ./models/")
print("📈 Visualizaciones guardadas en: ./output/")
print("\n" + "="*60)